# Cross-Entropy Loss in PyTorch

## Learning objectives

By the end of this notebook, you should be able to explain:

1. What **Binary Cross-Entropy (BCE)** is.
2. What **Cross-Entropy Loss (CE)** is.
3. When to use BCE vs CE.
4. Why PyTorch loss functions usually expect **logits**.
5. Why `argmax` is useful for prediction but not for training.
6. How logits, probabilities, loss, gradients, and weight updates fit together.



## 1. Binary Cross-Entropy Loss

Binary Cross-Entropy is used when each output represents a **binary yes/no decision**.

For one prediction:

$$
L = -\left[y\log(p) + (1-y)\log(1-p)\right]
$$

where:

- $y \in \{0,1\}$ is the true target
- $p$ is the predicted probability of class 1

If $y=1$:

$$
L=-\log(p)
$$

If $y=0$:

$$
L=-\log(1-p)
$$

So BCE strongly penalizes predictions that are **confidently wrong**.


In [1]:
import torch
import torch.nn.functional as F

# True class = 1
target = torch.tensor([1.0])

for p in [0.99, 0.8, 0.5, 0.1, 0.01]:
    probability = torch.tensor([p])
    loss = F.binary_cross_entropy(probability, target)
    print(f"p={p:>4}  BCE loss={loss.item():.4f}")

p=0.99  BCE loss=0.0101
p= 0.8  BCE loss=0.2231
p= 0.5  BCE loss=0.6931
p= 0.1  BCE loss=2.3026
p=0.01  BCE loss=4.6052


### BCE is not limited to datasets with only two classes

The word **binary** refers to each output being a binary decision.

### Binary classification

Example:

> Is this email spam?

One output:

$$
P(	ext{spam})
$$

### Multi-label classification

An image may simultaneously contain:

```text
dog     = 1
cat     = 0
car     = 1
person  = 1
tree    = 0
```

Each label is an independent yes/no question.

Therefore BCE is appropriate for both:

- binary classification
- multi-label classification


In [2]:
# Example: multi-label classification

targets = torch.tensor([
    [1., 0., 1., 1., 0.]
])

logits = torch.tensor([
    [2.3, -1.4, 1.2, 3.0, -0.8]
])

loss = F.binary_cross_entropy_with_logits(logits, targets)

print("Logits:", logits)
print("Probabilities:", torch.sigmoid(logits))
print("BCEWithLogitsLoss:", loss.item())

Logits: tensor([[ 2.3000, -1.4000,  1.2000,  3.0000, -0.8000]])
Probabilities: tensor([[0.9089, 0.1978, 0.7685, 0.9526, 0.3100]])
BCEWithLogitsLoss: 0.19978663325309753


## 2. Cross-Entropy Loss

Cross-Entropy is commonly used for **multi-class classification** when exactly one class is correct.

Example:

```text
0 = cat
1 = dog
2 = horse
```

Suppose the correct class is `dog`.
$$
\mathbf{y} = [0,\;1,\;0]
$$
If the predicted probabilities are:

$$
\mathbf{p} = [0.10,\;0.80,\;0.10]
$$

then:

$$
L=-\log(0.80)
$$

If the model assigns only 0.01 probability to the correct class:

$$
L=-\log(0.01)
$$

the loss becomes much larger.

For one-hot targets, cross-entropy can be written as:

$$
L=-\sum_i <\mathbf{y}_i, \log(\mathbf{p}_i)>
$$


$$
L=-\sum_i y_i\log(p_i)
$$

Only the probability of the correct class contributes directly to the final value.


In [3]:
import math

for p_correct in [0.99, 0.8, 0.5, 0.1, 0.01]:
    loss = -math.log(p_correct)
    print(f"P(correct)={p_correct:>4}  CE loss={loss:.4f}")

P(correct)=0.99  CE loss=0.0101
P(correct)= 0.8  CE loss=0.2231
P(correct)= 0.5  CE loss=0.6931
P(correct)= 0.1  CE loss=2.3026
P(correct)=0.01  CE loss=4.6052


## 3. What are logits?

A neural network normally outputs **logits**.

Example:

$$
[2.0,\;1.0,\;0.1]
$$

Logits:

- are raw model outputs
- can be negative or positive
- do not need to be between 0 and 1
- do not need to sum to 1

For multi-class classification, **softmax** converts logits to probabilities:

$$
p_i = \frac{e^{{z_i}}}{\sum_j e^{z_j}}
$$


$$
log(p_i) = log(\frac{e^{z_i}}{\sum_j e^{z_j}}) = log(e^{z_i}) - log(\sum_j e^{z_j}) = z_i + logsumexp(...)
$$


**operator: logsumexp**

In [4]:
logits = torch.tensor([[2.0, 1.0, 0.1]])

probabilities = torch.softmax(logits, dim=1)

print("Logits:", logits)
print("Probabilities:", probabilities)
print("Sum:", probabilities.sum(dim=1))

Logits: tensor([[2.0000, 1.0000, 0.1000]])
Probabilities: tensor([[0.6590, 0.2424, 0.0986]])
Sum: tensor([1.0000])


## 4. Why does `CrossEntropyLoss` use logits?

In PyTorch, this is the correct pattern:

```python
logits = model(x)
loss = torch.nn.CrossEntropyLoss()(logits, labels)
```

Do **not** apply softmax first:

```python
# Not recommended
probabilities = torch.softmax(logits, dim=1)
loss = torch.nn.CrossEntropyLoss()(probabilities, labels)
```

Conceptually:

```text
logits
   ↓
LogSoftmax
   ↓
Negative Log-Likelihood
   ↓
Cross-Entropy Loss
```

PyTorch combines these operations internally.


### Why combine softmax and cross-entropy?

The main reason is **numerical stability**.

Suppose the logits are:

$$
[1000,\;999,\;998]
$$

A naive implementation of softmax would try to compute:

$$
e^{1000}
$$

which can overflow.

PyTorch uses a mathematically equivalent, numerically stable implementation based on ideas such as the **log-sum-exp trick**.

So we should give `CrossEntropyLoss` the original logits.


In [ ]:
logits = torch.tensor([
    [1000.0, 999.0, 998.0]
])

target = torch.tensor([0])

criterion = torch.nn.CrossEntropyLoss()
loss = criterion(logits, target)

print("Loss:", loss.item())
print("Stable probabilities:", torch.softmax(logits, dim=1))

## 5. BCE has the same idea

For binary classification, a model can output one logit:

$$
z
$$

Sigmoid converts it to a probability:

$$
p = \sigma(z)= \frac{1}{1+e^{-z}}
$$

Conceptually:

```text
logit
  ↓
sigmoid
  ↓
probability
  ↓
Binary Cross-Entropy
```

You could write:

```python
probability = torch.sigmoid(logit)
loss = torch.nn.BCELoss()(probability, target)
```

But the recommended PyTorch version is:

```python
loss = torch.nn.BCEWithLogitsLoss()(logit, target)
```

It combines sigmoid and BCE in a more numerically stable way.


In [ ]:
logits = torch.tensor([2.5])
target = torch.tensor([1.0])

loss = torch.nn.BCEWithLogitsLoss()(logits, target)

print("Logit:", logits.item())
print("Probability:", torch.sigmoid(logits).item())
print("Loss:", loss.item())

## 6. BCE vs Cross-Entropy

| Problem type | Example target | Output interpretation | Typical loss |
|---|---|---|---|
| Binary classification | `1` | one yes/no decision | `BCEWithLogitsLoss` |
| Multi-label classification | `[1,0,1,1]` | multiple independent yes/no decisions | `BCEWithLogitsLoss` |
| Multi-class classification | `2` | exactly one class is correct | `CrossEntropyLoss` |

### Key intuition

**Cross-Entropy:** classes compete with each other.

```text
cat OR dog OR horse
```

**Binary Cross-Entropy:** each output is its own binary question.

```text
dog?     yes/no
person?  yes/no
car?     yes/no
```


## 7. Logits vs predicted classes

Suppose the model outputs:

```text
[2.1, -0.5, 4.3]
```

These are **logits**.

The predicted class is:

```python
predicted_class = logits.argmax()
```

which gives:

```text
2
```

`argmax` keeps only the index of the largest value.

That is useful for **prediction**, but not for training.


In [ ]:
logits = torch.tensor([
    [2.1, -0.5, 4.3],
    [1.2,  3.8, 0.7]
])

predicted_classes = logits.argmax(dim=1)

print("Logits:")
print(logits)
print()
print("Predicted classes:", predicted_classes)

## 8. Why is `argmax` not enough for training?

Compare:

```text
Model A: [0.1, 0.2, 0.3]
Model B: [0.1, 0.2, 10.0]
```

Both predict class `2`:

```python
argmax(...) == 2
```

But their confidence is very different.

`argmax`:

1. throws away most of the information in the logits
2. is not differentiable

Training requires a differentiable loss so gradients can flow backward through the network.

Therefore:

```python
loss = criterion(logits, labels)
loss.backward()
```

is used during training.


## 9. Training pipeline

A typical PyTorch training step is:

```python
optimizer.zero_grad()

logits = model(x)
loss = criterion(logits, y)

loss.backward()
optimizer.step()
```

The roles are:

```text
model(x)               → computes logits
criterion(logits, y)   → computes loss
loss.backward()        → computes gradients
optimizer.step()       → changes weights
```

`optimizer.zero_grad()` clears gradients from the previous iteration because PyTorch **accumulates gradients by default**.


## 10. Quick check

Answer these before revealing the answers:

1. Should we apply softmax before `CrossEntropyLoss`?
2. Which loss would you use for an image that may contain both a dog and a person?
3. Which loss would you use when an image must be exactly one of 10 classes?
4. Why is `argmax` unsuitable for training?
5. Which operation computes gradients?
6. Which operation changes the weights?


### Answers

1. **No.** Give raw logits directly to `CrossEntropyLoss`.
2. **`BCEWithLogitsLoss`**, because the labels are independent and multiple labels may be true.
3. **`CrossEntropyLoss`**, because exactly one class is correct.
4. `argmax` discards confidence information and is not differentiable.
5. **`loss.backward()`** computes gradients.
6. **`optimizer.step()`** updates the model parameters.


## Final mental model

```text
MULTI-CLASS
───────────
model
  ↓
logits
  ↓
CrossEntropyLoss
  ↓
loss
  ↓
backward()
  ↓
gradients
  ↓
optimizer.step()
  ↓
updated weights


BINARY / MULTI-LABEL
────────────────────
model
  ↓
logits
  ↓
BCEWithLogitsLoss
  ↓
loss
  ↓
backward()
  ↓
gradients
  ↓
optimizer.step()
  ↓
updated weights
```

During inference:

```python
# Multi-class probabilities
probs = torch.softmax(logits, dim=1)

# Multi-class prediction
predicted_class = logits.argmax(dim=1)

# Binary / multi-label probabilities
probs = torch.sigmoid(logits)
```

### One-sentence takeaway

> **Train with logits and a differentiable loss; convert to probabilities or classes when you need predictions.**
